# 01 - Data Collection & Validation
**ZHAW AI-Applications: Skin Lesion Classifier**

Downloads and validates the HAM10000 dataset.

**HAM10000 Classes:**
- mel: Melanoma
- nv: Melanocytic nevi
- bcc: Basal cell carcinoma
- akiec: Actinic keratoses
- bkl: Benign keratosis
- df: Dermatofibroma
- vasc: Vascular lesions

In [ ]:
import os, sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
from dotenv import load_dotenv
load_dotenv('../.env')
print('Environment loaded')

In [ ]:
# Install requirements if needed
# !pip install -r ../requirements.txt

## Option A: Kaggle API Download

In [ ]:
# Configure Kaggle credentials
import json
from pathlib import Path

kaggle_dir = Path.home() / '.kaggle'
kaggle_dir.mkdir(exist_ok=True)

creds = {
    'username': os.getenv('KAGGLE_USERNAME', 'YOUR_USERNAME'),
    'key': os.getenv('KAGGLE_KEY', 'YOUR_KEY')
}
cred_file = kaggle_dir / 'kaggle.json'
cred_file.write_text(json.dumps(creds))
cred_file.chmod(0o600)
print(f'Credentials saved to {cred_file}')

In [ ]:
import subprocess

# Download HAM10000
result = subprocess.run(
    ['kaggle', 'datasets', 'download', '-d', 'kmader/skin-cancer-mnist-ham10000',
     '-p', '../data/raw/', '--unzip'],
    capture_output=True, text=True
)
print('STDOUT:', result.stdout)
print('STDERR:', result.stderr)
print('Return code:', result.returncode)

## Option B: Manual Download
1. Go to: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
2. Download and extract to `data/raw/`

## Validate Dataset

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

raw_dir = Path('../data/raw')

# Check metadata
meta = pd.read_csv(raw_dir / 'HAM10000_metadata.csv')
print(f'Total samples: {len(meta)}')
print(f'Columns: {list(meta.columns)}')
print('\nClass distribution:')
print(meta['dx'].value_counts())

In [ ]:
# Count images
imgs_p1 = list((raw_dir / 'ham10000_images_part_1').glob('*.jpg'))
imgs_p2 = list((raw_dir / 'ham10000_images_part_2').glob('*.jpg'))
print(f'Part 1: {len(imgs_p1)} images')
print(f'Part 2: {len(imgs_p2)} images')
print(f'Total: {len(imgs_p1) + len(imgs_p2)} images')

In [ ]:
# Sample image display
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(2, 7, figsize=(20, 6))
classes = meta['dx'].unique()

all_imgs = {p.stem: p for p in imgs_p1 + imgs_p2}

for i, cls in enumerate(sorted(classes)):
    sample = meta[meta['dx']==cls].sample(2, random_state=42)
    for j, (_, row) in enumerate(sample.iterrows()):
        img_path = all_imgs.get(row['image_id'])
        if img_path:
            img = Image.open(img_path)
            axes[j][i].imshow(img)
            axes[j][i].set_title(f'{cls}', fontsize=8)
            axes[j][i].axis('off')

plt.suptitle('HAM10000 - Sample Images per Class', fontsize=14)
plt.tight_layout()
plt.savefig('../docs/screenshots/sample_images.png', dpi=150)
plt.show()

In [ ]:
# Run full preparation pipeline
import subprocess
result = subprocess.run(
    ['python', '-m', 'src.data.prepare_ham10000'],
    capture_output=True, text=True, cwd='..'
)
print(result.stdout)
if result.returncode != 0:
    print('ERROR:', result.stderr)